# Vehicle Tutorial notebookified
This is an easy environment to test, run and use vehicle, following the [vehicle tutorial](https://vehicle-lang.github.io/tutorial/).

Each question has a brief description, but for more detail refer to the tutorial. Some questions from the tutorial have been skipped or moved about, but it will be clear which part of the tutorial each section is referring to.

## Setup
Check vehicle and marabou are installed, download supplementary materials and setup helpers for the notebook. Do not edit these cells.

In [ ]:
%%bash
pip install vehicle-lang maraboupy
wget -nc -q --show-progress https://github.com/vehicle-lang/vehicle/raw/refs/heads/dev/examples/windController/controller.onnx
wget -nc -q --show-progress https://github.com/vehicle-lang/vehicle/raw/refs/heads/dev/examples/windController/windController.vcl
wget -nc -q --show-progress https://github.com/vehicle-lang/tutorial/raw/refs/heads/tutorial/exercises/acasXu/acasXu_1_7.onnx
wget -nc -q --show-progress https://github.com/vehicle-lang/tutorial/raw/refs/heads/tutorial/exercises/acasXu/acasXu_1_8.onnx
wget -nc -q --show-progress https://github.com/vehicle-lang/tutorial/raw/refs/heads/tutorial/exercises/acasXu/acasXu_1_9.onnx
wget -nc -q --show-progress https://github.com/vehicle-lang/tutorial/raw/refs/heads/tutorial/exercises/MNIST-incomplete/mnist-classifier.onnx
wget -nc -q --show-progress https://github.com/vehicle-lang/tutorial/raw/refs/heads/tutorial/exercises/MNIST-incomplete/t2-images.idx
wget -nc -q --show-progress https://github.com/vehicle-lang/tutorial/raw/refs/heads/tutorial/exercises/MNIST-incomplete/t2-labels.idx

In [ ]:
from IPython.core.magic import register_cell_magic
from pathlib import Path
import shutil

@register_cell_magic
def vcl(line, cell):
    p = Path(line)
    l = p.stem + "-incomplete" + p.suffix
    shutil.copy(l, line)
    with open(line, "a") as f:
        f.write(cell)
    print(f"Written to {line}")

# [Section 1](https://vehicle-lang.github.io/tutorial/chapters/language/): Learning the basics

This section covers chapter 1 of the tutorial, an introduction to using vehicle.

## 1.1. [Exercise (⭑)](https://vehicle-lang.github.io/tutorial/chapters/language/#exercise-)

This exercise is to run vehicle on the partially complete specification.

To do so, look over the specification below, and run the cell to save it to the file `windController.vcl`. The first task is to invoke vehicle to check the network `controller.onnx` against the specification using Marabou. Use the `%%bash` box to type commands to do with vehicle.

If everything has been setup properly, you should see `no counterexample exist`.

This example is only used to make sure vehicle is working and you understand how to use it before it gets more complicated. If you are interested in where this example comes from and how it fits into the vehicle tutorial, come back after going through the rest of the tutorial and read [this section](https://vehicle-lang.github.io/tutorial/chapters/exporting-to-agda/).

In [ ]:
%%writefile windController.vcl
--------------------------------------------------------------------------------
-- Inputs and outputs

type InputVector = Tensor Real [2]

currentSensor  = 0
previousSensor = 1

type OutputVector = Tensor Real [1]

velocity = 0

--------------------------------------------------------------------------------
-- Network

@network
controller : InputVector -> OutputVector

-- Normalises the input values from the range [-4, 4] metres to the range [0,1]
normalise : InputVector -> InputVector
normalise x = foreach i . (x ! i + 4.0) / 8.0

--------------------------------------------------------------------------------
-- Safety property

safeInput : InputVector -> Bool
safeInput x = forall i . -3.25 <= x ! i <= 3.25

safeOutput : InputVector -> Bool
safeOutput x = let y = controller (normalise x) ! velocity in
  -1.25 < y + 2 * (x ! currentSensor) - (x ! previousSensor) < 1.25

@property
safe : Bool
safe = forall x . safeInput x => safeOutput x

In [ ]:
%%bash
vehicle verify

## 1.2 [Exercise (⭑⭑). Problem Space versus Input/Output Space](https://vehicle-lang.github.io/tutorial/chapters/language/#exercise--problem-space-versus-inputoutput-space)

The rest of this section follows AcasXu as described in the lectures.

The following cell contains the preamble for the vehicle file. Any edit you make to this cell will be reflected in all future AcasXu files. Start by giving definitions to the functions at the end of the next cell.

**Note:** Depending on how you write the specificaiton, you may get a warning about strict and non-strict inequalities. For the purposes of these exercises, this can be safely ignored. If the warning is annoying you, then you can add `--no-warnings` at the _**end**_ of the command.

In [ ]:
%%writefile acasXu-incomplete.vcl
--------------------------------------------------------------------------------
-- Full specification of the ACAS XU networks

-- Taken from Appendix VI of "Reluplex: An Efficient SMT Solver for Verifying
-- Deep Neural Networks" at https://arxiv.org/pdf/1702.01135.pdf

-- Comments describing the properties are taken directly from the text.

--------------------------------------------------------------------------------
-- Utilities

-- The value of the constant `pi`.
pi = 3.141592

--------------------------------------------------------------------------------
-- Inputs

-- We first define a new name for the type of inputs of the network.
-- In particular, it takes inputs of the form of a vector of 5 rational numbers.

type Input = Tensor Real [5]

-- Next we add meaningful names for the indices.
-- The fact that all vector types come annotated with their size means that it
-- is impossible to mess up indexing into vectors, e.g. if you changed
-- `distanceToIntruder = 0` to `distanceToIntruder = 5` the specification would
-- fail to type-check.

distanceToIntruder = 0   -- measured in metres
angleToIntruder    = 1   -- measured in radians
intruderHeading    = 2   -- measured in radians
speed              = 3   -- measured in metres/second
intruderSpeed      = 4   -- measured in meters/second

--------------------------------------------------------------------------------
-- Outputs

-- Outputs are also a vector of 5 rationals. Each one representing the score
-- for the 5 available courses of action.

type Output = Tensor Real [5]

-- Again we define meaningful names for the indices into output vectors.

clearOfConflict = 0
weakLeft        = 1
weakRight       = 2
strongLeft      = 3
strongRight     = 4

--------------------------------------------------------------------------------
-- The network

-- Next we use the `network` annotation to declare the name and the type of the
-- neural network we are verifying. The implementation is passed to the compiler
-- via a reference to the ONNX file at compile time.

@network
acasXu : Input -> Output

--------------------------------------------------------------------------------
-- Normalisation

-- As is common in machine learning, the network operates over
-- normalised values, rather than values in the problem space
-- (e.g. using standard units like m/s).
-- This is an issue for us, as we would like to write our specification in
-- terms of the problem space values .
-- Therefore before applying the network, we first have to normalise
-- the values in the problem space.

-- For clarity, we therefore define a new type synonym
-- for unnormalised input vectors which are in the problem space.
type UnnormalisedInput = Tensor Real [5]

-- Next we define the minimum and maximum values that each input can take.
-- These correspond to the range of the inputs that the network is designed
-- to work over.
minimumInputValues : UnnormalisedInput
minimumInputValues = [0,0,0,0,0]

maximumInputValues : UnnormalisedInput
maximumInputValues = [60261.0, 2*pi, 2*pi, 1100.0, 1200.0]

-- We can therefore define a simple predicate saying whether a given input
-- vector is in the right range.
validInput : UnnormalisedInput -> Bool
validInput x = forall i . minimumInputValues ! i <= x ! i <= maximumInputValues ! i

-- Then the mean values that will be used to scale the inputs.
meanScalingValues : UnnormalisedInput
meanScalingValues = [19791.091, 0.0, 0.0, 650.0, 600.0]

-- We can now define the normalisation function that takes an input vector and
-- returns the unnormalised version.
normalise : UnnormalisedInput -> Input
normalise x = foreach i .
  (x ! i - meanScalingValues ! i) / (maximumInputValues ! i)

-- Using this we can define a new function that first normalises the input
-- vector and then applies the neural network.
normAcasXu : UnnormalisedInput -> Output
normAcasXu x = acasXu (normalise x)

-- A constraint that says the network chooses output `i` when given the
-- input `x`. We must necessarily provide a finite index that is less than 5
-- (i.e. of type Index 5). The `a ! b` operator lookups index `b` in vector `a`.
advises : Index 5 -> UnnormalisedInput -> Bool
advises i x = forall j . i != j => normAcasXu x ! i < normAcasXu x ! j


minimalScore : Index 5 -> UnnormalisedInput -> Bool
minimalScore i x = TODO

directlyAhead : TODO

## Property 3.

The goal is to implement property 3 as described by [Katz et al.](https://arxiv.org/pdf/1702.01135):

* Description: If the intruder is directly ahead and is moving towards the
ownship, the score for COC will not be minimal.
* Tested on: all networks except N1,7, N1,8, and N1,9.
* Input constraints: 1500 ≤ ρ ≤ 1800, −0.06 ≤ θ ≤ 0.06, ψ ≥ 3.10, vown ≥ 980,
vint ≥ 960.
( Desired output property: the score for COC is not the minimal score.

Edit the partial file below to fill in the `property 3` section, then use the `%%bash` section to run the verification of the specification `acasXu.vcl` against `acasXu_1_7.onnx`, `acasXu_1_8.onnx` and `acasXu_1_9.onnx`.

__Note that these are not necessarily meant to pass__, but they should still reflect the description of the property. What does it mean if they do or don't pass? What situations does it affect?

In [ ]:
%%vcl acasXu.vcl
--------------------------------------------------------------------------------
-- Property 3 - Do it yourself!

-- If the intruder is directly ahead and is moving towards the
-- ownship, the score for COC will not be minimal.

movingTowards : TODO

@property
property3 : Bool
property3 = TODO

In [ ]:
%%bash
vehicle verify

## 1.3. [Exercise (**)](https://vehicle-lang.github.io/tutorial/chapters/language/#exercise--the-full-acas-xu-challenge-in-one-file). More properties

There are some more missing properties and definitions. The task is to fill in each property based on its definition.

Similarly to above, these are not required to pass the verification.

## Property 4

* Description: If the intruder is directly ahead and is moving away from the
ownship but at a lower speed than that of the ownship, the score for COC
will not be minimal.
* Tested on: all networks except N1,7, N1,8, and N1,9.
* Input constraints: 1500 ≤ ρ ≤ 1800, −0.06 ≤ θ ≤ 0.06, ψ = 0, vown ≥ 1000,
700 ≤ vint ≤ 800.
* Desired output property: the score for COC is not the minimal score.

In [ ]:
%%vcl acasXu.vcl
--------------------------------------------------------------------------------
-- Property 4

-- If the intruder is directly ahead and is moving away from the
-- ownship but at a lower speed than that of the ownship, the score for COC
-- will not be minimal.

movingAway : TODO

@property
property4 : Bool
property4 = TODO

In [ ]:
%%bash
vehicle verify

## Property 5

* Description: If the intruder is near and approaching from the left, the network
advises “strong right”.
* Tested on: N1,1.
* Input constraints: 250 ≤ ρ ≤ 400, 0.2 ≤ θ ≤ 0.4, −3.141592 ≤ ψ ≤ 3.141592 + 0.005, 100 ≤ vown ≤ 400, 0 ≤ vint ≤ 400.
* Desired output property: the score for “strong right” is the minimal score.

In [ ]:
%%vcl acasXu.vcl
--------------------------------------------------------------------------------
-- Property 5

-- If the intruder is near and approaching from the left, the network
-- advises “strong right”.

nearAndApproachingFromLeft : TODO

@property
property5 : Bool
property5 = TODO

In [ ]:
%%bash
vehicle verify

# 2. [MNIST Robustness](https://vehicle-lang.github.io/tutorial/chapters/proving-robustness/)

Following the next chapter of the tutorial into robustness of neural networks, we look over specifications that define such properties. As with before, the cell below represents the base file that you are free to edit. All following tasks will inherit this, and add to the end of this file for each task.

MNIST is a dataset of hand drawn images of numbers that is a classic machine learning problem. Each image in the dataset is a 28 by 28 pixel square with only 1 colour channel.

# 2.1: Define traditional robustness (**)

_Note that this is not an exercise for the vehicle tutorial directly, but [the chapter](https://vehicle-lang.github.io/tutorial/chapters/proving-robustness/) will still
provide some insight._

The next task is to define what robustness is and to create a property that can prove a network is robust. Use vehicle in the cell below the next to test your implementation.

Start by filling in the gaps in the preamble, by defining the type of an image, what is a valid image, a function checking if the network advised a specific label, and what it means to be robust around a specific image.

In [ ]:
%%writefile mnist-robustness-incomplete.vcl
--------------------------------------------------------------------------------
-- Inputs and outputs

-- Define the type for our input images. Note that the input is two-dimensional

type Image = TODO

-- We define the type of the output labels
-- i.e a number between 0 and 9, one for each digit

type Label = Index 10

-- Define a predicate that states that all the pixel values in a given image are in the range 0.0 to 1.0

validImage : Image -> Bool
validImage x = TODO

--------------------------------------------------------------------------------
-- Network

-- Declare the network used to classify images. The output of the network is a
-- score for each of the digits 0 to 9.
@network
classifier : Image -> Tensor Real [10]

-- The classifier advises that input image `x` has label `i` if the score
-- for label `i` is greater than the score of any other label `j`.

advises : Image -> Label -> Bool
advises x i = TODO

--------------------------------------------------------------------------------
-- Definition of robustness around a point

-- First we define the parameter `epsilon` that will represent the radius of the ball that we want the network to be robust in. Note that we declare this as a parameter which allows the value of `epsilon` to be specified at compile time rather than be fixed in the specification.

@parameter
epsilon : Real

-- Next we define what it means for an image `x` to be in a ball of
-- size epsilon around 0.

boundedByEpsilon : Image -> Bool
boundedByEpsilon x = forall i j . -epsilon <= x ! i ! j <= epsilon

-- We now define what it means for the network to be robust around an image `x`  that should be classified as `y`. Namely, that for any perturbation no greater than epsilon then if the perturbed image is still a valid image then the network should still advise label `y` for the perturbed version of `x`.

robustAround : Image -> Label -> Bool
robustAround image label = forall pertubation .
  let perturbedImage = image - pertubation in
  boundedByEpsilon pertubation and validImage perturbedImage =>
    TODO

Following on, next define the property robust, and check the neural network `mnist-classifier.onnx` against the specification, with the chosen images `t2-images.idx` and labels `t2-labels.idx`. Pick `epsilon` to be 0.005 or lower. If the task is timing out or having issues, then reduce `epsilon`.

In [ ]:
%%vcl mnist-robustness.vcl
--------------------------------------------------------------------------------
-- Robustness with respect to a dataset

-- We only really care about the network being robust on the set of images it will encounter. Indeed it is much more challenging to expect the network to be robust around all possible images. After all most images will be just be random noise.

-- Unfortunately we can't characterise the set of "reasonable" input images.
-- Instead we approximate it using the training dataset, and ask that the
-- network is robust around images in the training dataset.

-- We first specify parameter `n` the size of the training dataset. Unlike
-- the earlier parameter `epsilon`, we set the `infer` option of the
-- parameter `n` to 'True'. This means that it does not need to be provided
--  manually but instead will be automatically inferred by the compiler.
-- In this case it will be inferred from the training datasets.

@parameter(infer=True)
n : Nat

-- We next declare two datasets, the training images and the corresponding
-- training labels. Note that we use the previously declared parameter `n`
-- to enforce that they are the same size.
@dataset
trainingImages : Vector Image n

@dataset
trainingLabels : Vector Label n

-- We then say that the network is robust if it is robust around every pair
-- of input images and output labels. Note the use of the `foreach`
-- keyword when quantifying over the index `i` in the dataset. Whereas `forall`
-- would return a single `Bool`, `foreach` constructs a `Vector` of booleans,
-- ensuring that Vehicle will report on the verification status of each image in
-- the dataset separately. If `forall` was omitted, Vehicle would only
-- report if the network was robust around *every* image in the dataset, a
-- state of affairs which is unlikely to be true.

@property
robust : Vector Bool n
robust = TODO

In [ ]:
%%bash
vehicle verify

# 2.2 Lipschitz Robustness (***)

There are alternative ideas of robustness like Lipschitz robustness, that can also be expressed in vehicle.

Try defining Lipschitz robustness.

**Note:** currently, Marabou cannot handle Lipschitz robustness so the vehicle specification cannot be used to verify. However, it would still be usable with the training backend, and can still be typechecked. To complete this exercise, write the lipscitz robustness and use the vehicle typechecker to verify that the vehicle file could be used with a sufficient verifier.

In [ ]:
%%vcl mnist-robustness.vcl

@parameter
lipschitzConstant : Real

@property
lipschitzRobustness : TODO

In [ ]:
%%bash
vehicle typecheck -s mnist-robustness.vcl
vehicle verify -s mnist-robustness.vcl -v Marabou -p lipschitzConstant:1 -p epsilon:0.005 -n classifier:mnist-classifier.onnx